# BraTS-PEDs 2026 — SwinUNETR: **NET Subregion** (Binary: BG vs NET)
**Label map (binary):** `0`=Background · `1`=NET (Non-Enhancing Tumor core, original label 2)

**Architecture choices for NET:**
- **T1CE + T1n emphasized** — NET appears iso/hypointense on T1CE (no enhancement), discriminated by comparing T1n vs T1CE; T2w/FLAIR also loaded for infiltrative context
- **Focal Tversky Loss (FTL) + Weighted BCE** — handles severe class imbalance (~0.1–1% prevalence), β=0.7 penalizes False Negatives heavily (NET is clinically critical but hard to detect)
- **SwinUNETR** — shifted-window self-attention learns long-range dependencies across T1n/T1CE contrast difference and tumor core geometry
- **T1CE-vs-T1n Contrast Augmentation** — simulates variable enhancement to differentiate NET from ET (label 1) and distinguish from CC (label 3)

Run all cells top to bottom. Only edit **Cell 2 — Configuration**.


In [12]:
# ── stdlib ────────────────────────────────────────────────────────────────────
import os, time, random, warnings
from pathlib import Path
from datetime import datetime
from functools import partial
warnings.filterwarnings("ignore")

# ── third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from tqdm import tqdm

# ── torch ─────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

torch.backends.cudnn.benchmark = True

import logging
torch._logging.set_logs(recompiles=False)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)

# ── MONAI ─────────────────────────────────────────────────────────────────────
from monai.config import print_config
from monai.utils import set_determinism
from monai.utils.enums import MetricReduction
from monai.data import (
    SmartCacheDataset, DataLoader, Dataset,
    decollate_batch, pad_list_data_collate,
)
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, CropForegroundd,
    RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
    RandShiftIntensityd, RandScaleIntensityd, RandGaussianNoised,
    RandAdjustContrastd, RandGaussianSmoothd,
    Activations, AsDiscrete, MapTransform,
)
from monai.networks.nets import UNet, SwinUNETR
from monai.networks.layers import Norm
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.inferers import sliding_window_inference

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print_config()
print(f"\nPyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"GPUs found: {NUM_GPUS}")
for i in range(NUM_GPUS):
    vram = round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({vram} GB)")


MONAI version: 1.5.0
Numpy version: 2.0.2
Pytorch version: 2.6.0+cu124
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: d388d1c6fec8cb3a0eebee5b5a0b9776ca59ca83
MONAI __file__: /home/cbme/phd/<username>/.local/lib/python3.9/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: 0.4.11
ITK version: 5.4.4
Nibabel version: 5.3.2
scikit-image version: 0.24.0
scipy version: 1.13.1
Pillow version: 11.2.1
Tensorboard version: 2.19.0
gdown version: 5.2.0
TorchVision version: 0.21.0+cu124
tqdm version: 4.66.4
lmdb version: 1.6.2
psutil version: 7.0.0
pandas version: 2.3.0+4.g1dfc98e16a
einops version: 0.8.1
transformers version: 4.40.2
mlflow version: 3.1.0
pynrrd version: 1.1.3
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies


PyTorch 2.6.0+cu124 | CUDA: True
GPUs f

## Cell 2 — Configuration
**Edit the two paths. Everything else is pre-tuned for NET subregion.**


In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PATHS                                           ║
# ╚══════════════════════════════════════════════════════════════╝
DATA_ROOT        = Path(r"/home/cbme/phd/bmz228464/Brats2026/Pred/Data/Deskullp_data/Train/")
EXPERIMENTS_ROOT = Path(r"/home/cbme/phd/bmz228464/Brats2026/Pred/Thulani/Model_T/")

MODEL_NAME = "SwinUNETR_NET_BinarySubregion"
MODEL_ROOT = EXPERIMENTS_ROOT / MODEL_NAME
RUN_NAME   = f"run_{datetime.now():%Y%m%d_%H%M}"
RUN_DIR    = MODEL_ROOT / RUN_NAME
for sub in ["", "figures", "predictions", "checkpoints"]:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)

SEED = 42
set_determinism(seed=SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SPLIT_RATIOS = (0.80, 0.20)

# ── NET-specific: T1CE + T1n are primary discriminators ──────────────────────
# NET (label 2) = Non-Enhancing Tumor core
#   T1CE : NET is iso/hypointense (NO enhancement) — contrast with ET which IS bright
#   T1n  : provides native T1 baseline to compute T1n vs T1CE difference
#   T2w  : NET is heterogeneous; T2 hypointense core helps boundary delineation
#   T2f  : FLAIR loaded for surrounding infiltrative context
IMAGE_KEYS = ["t1n", "t1c", "t2w", "t2f"]    # all 4 modalities — 4-channel input
NET_KEYS   = ["t2w", "t2f"]                    # primary NET contrast channels
ALL_KEYS   = IMAGE_KEYS + ["label"]

# Binary task: BG=0, NET=1
# Label remapping: original label 2 (NET) → 1; everything else → 0
IN_CHANNELS  = 4
OUT_CHANNELS = 2

# ── Patch & inference ─────────────────────────────────────────────────────────
# 96³ patch: NET is moderately sized (~1–5% volume in tumor cases), 96³ gives
# sufficient context to distinguish from adjacent ET (label 1) and CC (label 3)
PATCH_SIZE          = (64, 64, 64)
SW_ROI              = (64, 64, 64)
SW_BATCH            = 4
SW_OVERLAP          = 0.25
NUM_POS_NEG_SAMPLES = 5

# ── SwinUNETR architecture ────────────────────────────────────────────────────
# feature_size=48: standard BraTS-scale; img_size must match PATCH_SIZE (96³)
# PATCH_SIZE=96³ is divisible by 32 — satisfies SwinUNETR window partition constraint
SWIN_FEATURE_SIZE = 48
SWIN_DEPTHS       = (2, 2, 2, 2)
SWIN_NUM_HEADS    = (3, 6, 12, 24)
SWIN_DROP_RATE    = 0.0
SWIN_ATTN_DROP    = 0.0
SWIN_DROP_PATH    = 0.0

# ── Training ──────────────────────────────────────────────────────────────────
MAX_EPOCHS          = 200
VAL_INTERVAL        = 2
BATCH_SIZE          = 2 * max(NUM_GPUS, 1)
VAL_BATCH_SIZE      = 1
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-5
USE_AMP             = True
CACHE_RATE          = 0.5
EARLY_STOP_PATIENCE = 40

DL_NUM_WORKERS    = 0
CACHE_NUM_WORKERS = 0
BEST_MODEL_NAME   = "best_swinunetr_net.pth"

# ── Focal Tversky Loss hyperparameters ────────────────────────────────────────
# NET is frequently missed (FN) because it looks similar to BG on T1CE.
# β=0.7 > α=0.3 → penalize FN more aggressively than FP.
# γ=0.75 → focal factor concentrates gradient on hard boundary voxels.
FTL_ALPHA = 0.3
FTL_BETA  = 0.7
FTL_GAMMA = 0.75

# Combined loss blending
COMBINED_FTL_WEIGHT     = 0.60   # FTL: handles hard FN voxels at NET boundary
COMBINED_BCE_WEIGHT     = 0.40   # WeightedBCE: prevents class-collapse on tiny NET
NET_POS_WEIGHT          = 30.0   # BCE penalty: each missed NET voxel costs 30× BG

print(f"Input channels  : {IN_CHANNELS}  {IMAGE_KEYS}")
print(f"Output channels : {OUT_CHANNELS}  (BG=0, NET=1)")
print(f"Primary keys    : {NET_KEYS}  ← T1CE vs T1n contrast for NET detection")
print(f"FTL params      : alpha={FTL_ALPHA}, beta={FTL_BETA}, gamma={FTL_GAMMA}")
print(f"Loss blend      : FTL×{COMBINED_FTL_WEIGHT} + WeightedBCE×{COMBINED_BCE_WEIGHT} (pos_weight={NET_POS_WEIGHT})")
print(f"Device          : {DEVICE}")


Run directory: /home/cbme/phd/bmz228464/Brats2026/Pred/Thulani/Model_T/SwinUNETR_NET_BinarySubregion/run_20260608_1549
Input channels  : 4  ['t1n', 't1c', 't2w', 't2f']
Output channels : 2  (BG=0, NET=1)
Primary keys    : ['t1n', 't1c']  ← T1CE vs T1n contrast for NET detection
FTL params      : alpha=0.3, beta=0.7, gamma=0.75
Loss blend      : FTL×0.6 + WeightedBCE×0.4 (pos_weight=30.0)
Device          : cuda:0


## Cell 3 — Custom Modules: SwinUNETR Wrapper + Combined NET Loss

**Why SwinUNETR for NET?**  
NET sits directly adjacent to ET (label 1) and CC (label 3). SwinUNETR's shifted-window self-attention captures long-range dependencies across the full 96³ patch, enabling the model to simultaneously see the dark-on-T1CE NET core and the adjacent bright-on-T1CE ET boundary — the key discriminating context for NET detection.

**Why Combined FTL + WeightedBCE?**  
NET prevalence in PED cases is ~0.1–2%. FTL with β=0.7 recovers missed NET voxels (FN). WeightedBCE with pos_weight=30 prevents trivial all-background collapse, especially in early training epochs.


In [14]:
# ── SwinUNETR wrapper for NET ─────────────────────────────────────────────────
# Replaces CBAMUNet3D — identical call interface used in Cells 8, 11, 12.
# SwinUNETR shifted-window attention captures long-range context across the full
# 96³ patch, critical for discriminating NET from adjacent ET/CC subregions.
#
# img_size MUST match PATCH_SIZE (96³) so partition windows tile evenly.
# feature_size=48 → ~62 M params; use_checkpoint=True saves VRAM during training.

from monai.networks.nets import SwinUNETR as _SwinUNETR

class SwinUNETRWrapper(nn.Module):
    """
    Thin wrapper around MONAI SwinUNETR to expose the same
    __init__ signature used in training / test / reload cells.
    """
    def __init__(
        self,
        in_channels: int = 4,
        out_channels: int = 2,
        img_size=None,
        feature_size: int = 48,
        depths=(2, 2, 2, 2),
        num_heads=(3, 6, 12, 24),
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        dropout_path_rate: float = 0.0,
        use_checkpoint: bool = True,
    ):
        super().__init__()
        if img_size is None:
            img_size = PATCH_SIZE
        self.net = _SwinUNETR(
            #img_size           = img_size,
            in_channels        = in_channels,
            out_channels       = out_channels,
            feature_size       = feature_size,
            depths             = depths,
            num_heads          = num_heads,
            drop_rate          = drop_rate,
            attn_drop_rate     = attn_drop_rate,
            dropout_path_rate  = dropout_path_rate,
            use_checkpoint     = use_checkpoint,
            spatial_dims       = 3,
        )

    def forward(self, x):
        return self.net(x)


# Sanity check
_sw = SwinUNETRWrapper(
    in_channels=4, out_channels=2,
    img_size=PATCH_SIZE, feature_size=SWIN_FEATURE_SIZE,
    depths=SWIN_DEPTHS, num_heads=SWIN_NUM_HEADS,
)
with torch.no_grad():
    _out = _sw(torch.randn(1, 4, *PATCH_SIZE))
assert _out.shape == (1, 2, *PATCH_SIZE), f"Unexpected shape: {_out.shape}"
del _sw, _out
print(f"SwinUNETRWrapper sanity check passed ✓  (img_size={PATCH_SIZE})")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Focal Tversky Loss                                                         ║
# ║  TI = TP / (TP + α·FP + β·FN)   FTL = (1 - TI)^γ                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class FocalTverskyLoss(nn.Module):
    """
    Focal Tversky Loss for binary segmentation.

    Args:
        alpha : weight on False Positives  (default 0.3 — tolerate more FP)
        beta  : weight on False Negatives  (default 0.7 — penalize missed NET)
        gamma : focal exponent             (default 0.75 — focus on hard voxels)
        smooth: Laplace smoothing          (default 1e-6)

    Note: alpha + beta must equal 1.0
    """
    def __init__(self, alpha: float = 0.3, beta: float = 0.7,
                 gamma: float = 0.75, smooth: float = 1e-6):
        super().__init__()
        assert abs(alpha + beta - 1.0) < 1e-6, "alpha + beta must equal 1.0"
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B, 2, D, H, W)  — raw 2-class logits
        targets : (B, 1, D, H, W)  — integer labels {0, 1}
        """
        probs = F.softmax(logits, dim=1)[:, 1]       # (B, D, H, W) NET prob
        tgt   = (targets[:, 0] == 1).float()          # (B, D, H, W) binary NET mask

        probs = probs.contiguous().view(probs.size(0), -1)
        tgt   = tgt.contiguous().view(tgt.size(0), -1)

        TP = (probs * tgt).sum(dim=1)
        FP = (probs * (1.0 - tgt)).sum(dim=1)
        FN = ((1.0 - probs) * tgt).sum(dim=1)

        tversky_index = (TP + self.smooth) / (
            TP + self.alpha * FP + self.beta * FN + self.smooth
        )
        ftl = (1.0 - tversky_index) ** self.gamma
        return ftl.mean()


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Combined Loss: Focal Tversky + Weighted BCE                                ║
# ║                                                                             ║
# ║  Why two terms for NET?                                                     ║
# ║   FTL (β=0.7) → recovers missed NET voxels (FN) near ET/CC boundary         ║
# ║   WeightedBCE (pos_weight=30) → prevents class-collapse in early training   ║
# ║   Combined: FTL shapes hard boundaries; BCE stabilizes the gradient         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class CombinedNETLoss(nn.Module):
    """
    Focal Tversky Loss + Weighted BCE for NET subregion (~0.1–2% prevalence).

    Args:
        ftl_alpha   : FTL FP weight (default 0.3)
        ftl_beta    : FTL FN weight (default 0.7) — heavier because NET is missed
        ftl_gamma   : FTL focal exponent (default 0.75)
        pos_weight  : BCE weight on positive (NET) voxels (default 30.0)
        ftl_weight  : blending weight for FTL term (default 0.60)
        bce_weight  : blending weight for BCE term (default 0.40)
        smooth      : Laplace smoothing for Tversky (default 1e-6)
    """
    def __init__(self, ftl_alpha: float = 0.3, ftl_beta: float = 0.7,
                 ftl_gamma: float = 0.75, pos_weight: float = 30.0,
                 ftl_weight: float = 0.60, bce_weight: float = 0.40,
                 smooth: float = 1e-6):
        super().__init__()
        self.ftl       = FocalTverskyLoss(alpha=ftl_alpha, beta=ftl_beta,
                                          gamma=ftl_gamma, smooth=smooth)
        self.ftl_weight = ftl_weight
        self.bce_weight = bce_weight
        # Register pos_weight as buffer so it moves with .to(device)
        self.register_buffer("pw", torch.tensor(pos_weight))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B, 2, D, H, W)  — raw 2-class logits
        targets : (B, 1, D, H, W)  — integer labels {0, 1}
        """
        # ── Focal Tversky term ──────────────────────────────────────────────────
        ftl_loss = self.ftl(logits, targets)

        # ── Weighted BCE term ──────────────────────────────────────────────────
        probs = F.softmax(logits, dim=1)[:, 1]        # (B, D, H, W)
        tgt   = (targets[:, 0] == 1).float()           # (B, D, H, W)

        flat_p = probs.contiguous().view(probs.size(0), -1).clamp(1e-7, 1.0 - 1e-7)
        flat_t = tgt.contiguous().view(tgt.size(0), -1)

        bce = -(
            self.pw * flat_t * torch.log(flat_p)
            + (1.0 - flat_t) * torch.log(1.0 - flat_p)
        ).mean(dim=1)

        return self.ftl_weight * ftl_loss + self.bce_weight * bce.mean()


# Sanity checks
_loss_fn = CombinedNETLoss(
    ftl_alpha=FTL_ALPHA, ftl_beta=FTL_BETA, ftl_gamma=FTL_GAMMA,
    pos_weight=NET_POS_WEIGHT,
    ftl_weight=COMBINED_FTL_WEIGHT, bce_weight=COMBINED_BCE_WEIGHT,
)
_lv = _loss_fn(torch.randn(2, 2, 8, 8, 8), torch.randint(0, 2, (2, 1, 8, 8, 8)))
print(f"CombinedNETLoss sanity check: {_lv.item():.4f} ✓")
print(f"  FTL  α={FTL_ALPHA} β={FTL_BETA} γ={FTL_GAMMA}")
print(f"  BCE  pos_weight={NET_POS_WEIGHT}")
print(f"  Blend FTL×{COMBINED_FTL_WEIGHT} + BCE×{COMBINED_BCE_WEIGHT}")


SwinUNETRWrapper sanity check passed ✓  (img_size=(64, 64, 64))
CombinedNETLoss sanity check: 5.9604 ✓
  FTL  α=0.3 β=0.7 γ=0.75
  BCE  pos_weight=30.0
  Blend FTL×0.6 + BCE×0.4


## Cell 4 — Data Discovery
Scans `DATA_ROOT` for BraTS-PEDs cases. **NET binary label**: original label 2 → 1, all others → 0.


In [15]:
FILE_KEYS = ["t1n", "t1c", "t2w", "t2f"]

def find_modality(case_dir: Path, key: str):
    # Try looking for the standard pattern first
    hits = sorted(case_dir.glob(f"*-{key}.nii.gz"))
    if not hits:
        # Grab files containing the key, but FILTER OUT any mask files
        hits = sorted([
            p for p in case_dir.glob(f"*{key}*.nii*") 
            if "mask" not in p.name.lower()
        ])
    return str(hits[0]) if hits else None

data_dicts = []
skipped    = []

case_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])

for d in case_dirs:
    sid = d.name
    mod_paths = {key: find_modality(d, key) for key in FILE_KEYS}
    missing   = [k for k, v in mod_paths.items() if v is None]
    if missing:
        skipped.append((sid, f"missing modalities: {missing}"))
        continue

    seg_hits = sorted(d.glob("*-seg.nii.gz")) or sorted(d.glob("*seg*.nii*"))
    if not seg_hits:
        skipped.append((sid, "no seg file"))
        continue

    # Include all cases — model must learn BG-only cases too (specificity)
    entry = {key: mod_paths[key] for key in FILE_KEYS}
    entry["label"]      = str(seg_hits[0])
    entry["subject_id"] = sid
    data_dicts.append(entry)

print(f"Total cases found : {len(data_dicts)}")
print(f"Skipped           : {len(skipped)}")

from collections import Counter
reasons = Counter(reason for _, reason in skipped)
for reason, count in reasons.most_common():
    print(f"  {count:4d}  {reason}")

# Count NET presence
net_count    = 0
no_net_count = 0
for entry in data_dicts:
    seg_arr = nib.load(entry["label"]).get_fdata()
    if (seg_arr == 2).sum() > 0:
        net_count += 1
    else:
        no_net_count += 1

print(f"\nCases WITH    NET (label 2 present) : {net_count}")
print(f"Cases WITHOUT NET (label 2 absent)  : {no_net_count}")
print(f"Total                               : {len(data_dicts)}")

# Label sanity on first case
if data_dicts:
    seg_img = nib.load(data_dicts[0]["label"])
    seg_arr = seg_img.get_fdata()
    print(f"\nLabel sanity check — first case: {data_dicts[0]['subject_id']}")
    full_map = {0: "BG", 1: "ET", 2: "NET", 3: "CC", 4: "ED"}
    for lbl in np.unique(seg_arr).astype(int):
        name = full_map.get(lbl, "UNKNOWN")
        vox  = int((seg_arr == lbl).sum())
        print(f"  Label {lbl} ({name:>3s}): {vox:>10,} voxels → binary NET: {1 if lbl==2 else 0}")

    net_vox = int((seg_arr == 2).sum())
    total   = seg_arr.size
    pct     = net_vox / total * 100
    print(f"\nNET class balance (first case): {net_vox:,} / {total:,} = {pct:.3f}% NET")
    print("(Moderate imbalance → FTL β=0.7 + WeightedBCE pos_weight=30 critical)")


Total cases found : 274
Skipped           : 0

Cases WITH    NET (label 2 present) : 272
Cases WITHOUT NET (label 2 absent)  : 2
Total                               : 274

Label sanity check — first case: BraTS-PED-00001-000
  Label 0 ( BG):  8,799,333 voxels → binary NET: 0
  Label 1 ( ET):      6,204 voxels → binary NET: 0
  Label 2 (NET):     87,311 voxels → binary NET: 1
  Label 3 ( CC):        607 voxels → binary NET: 0
  Label 4 ( ED):     34,545 voxels → binary NET: 0

NET class balance (first case): 87,311 / 8,928,000 = 0.978% NET
(Moderate imbalance → FTL β=0.7 + WeightedBCE pos_weight=30 critical)


## Cell 5 — Transforms
- Binary remap: label 2 → 1, all others → 0
- **T1CE as foreground crop source** — tumor core (where NET lives) is delineated by T1CE
- **T1n / T1CE differential augmentation** — simulates variable gadolinium enhancement to harden the NET vs ET boundary
- `pos=4, neg=1` — strongly biases patches toward NET voxels
- `allow_smaller=True` — prevents crash when volume is near patch boundary


In [ ]:
from monai.transforms import Lambdad

NET_KEYS = ["t2f","t2w"]
# ── Training transforms ───────────────────────────────────────────────────────
train_transforms = Compose([
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes="RAS"),
    
    # Built-in robust remapping (maps 2 to 1, everything else to 0)
    Lambdad(keys=["label"], func=lambda x: (x == 2).long()),
    Spacingd(keys=ALL_KEYS,pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest"), padding_mode="border"),
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
    
    CropForegroundd(
        keys=ALL_KEYS,
        source_key="t1c",
        select_fn=lambda x: x > 0,
        allow_smaller=True,
        margin=10,
    ),
    
    RandCropByPosNegLabeld(
        keys=ALL_KEYS,
        label_key="label",
        spatial_size=list(PATCH_SIZE),
        pos=4,
        neg=1,
        num_samples=NUM_POS_NEG_SAMPLES,
        image_key="t1c",
        image_threshold=0,
        allow_smaller=True,
    ),
    
    # ── Spatial augmentations ─────────────────────────────────────────────────
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=0),
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=1),
    RandFlipd(keys=ALL_KEYS, prob=0.5, spatial_axis=2),
    RandRotate90d(keys=ALL_KEYS, prob=0.5, max_k=3),
    #RandAffined
    #RandZoomd
    # ── ED-SPECIFIC: T2/FLAIR intensity augmentations ─────────────────────────
    # Heavier augmentation on T2w and T2f only (primary ED channels)
    # Simulates scanner variability in fluid-tissue contrast
    #RandGaussianNoised(keys=IMAGE_KEYS, std=0.02, mean=0.0,prob=0.25),       # ↑ from 0.01
    # Gaussian blur on T2/FLAIR: mimics partial volume effects at edema boundary
    #RandGaussianSmoothd(keys=IMAGE_KEYS,sigma_x=(0.5, 1.5),sigma_y=(0.5, 1.5),sigma_z=(0.5, 1.5), prob=0.2),
    RandAdjustContrastd(keys=NET_KEYS, prob=0.3, gamma=(0.7, 1.5),),
    RandScaleIntensityd(keys=IMAGE_KEYS, factors=0.2, prob=0.4),
    RandShiftIntensityd(keys=IMAGE_KEYS, offsets=0.1, prob=0.4),
    #RandGibbsNoised
    #RandGammaCorrectiond(keys=ED_KEYS, gamma_range=(0.7, 1.4), prob=0.5),
])


# ── Val transforms ────────────────────────────────────────────────────────────
val_transforms = Compose([
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes="RAS"),
    
    # Use Lambdad here as well
    Lambdad(keys=["label"], func=lambda x: (x == 2).long()),
    
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
    CropForegroundd(
        keys=ALL_KEYS,
        source_key="t1n",
        select_fn=lambda x: x > 0,
        allow_smaller=True,
        margin=10,
    ),
])
test_transforms = val_transforms

## Cell 6 — Train / Val / Test Split


In [17]:
rng     = np.random.default_rng(SEED)
indices = np.arange(len(data_dicts))
rng.shuffle(indices)

n_total = len(indices)
n_train = int(round(n_total * SPLIT_RATIOS[0]))
n_val   = int(round(n_total * SPLIT_RATIOS[1]))

train_files = [data_dicts[i] for i in indices[:n_train]]
val_files   = [data_dicts[i] for i in indices[n_train : n_train + n_val]]
test_files  = [data_dicts[i] for i in indices[n_train + n_val:]]

splits_df = pd.DataFrame(
    [{"subject_id": d["subject_id"], "split": s}
     for files, s in [(train_files,"train"),(val_files,"val"),(test_files,"test")]
     for d in files]
).sort_values("subject_id").reset_index(drop=True)
splits_df.to_csv(RUN_DIR / "splits.csv", index=False)
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")


train=219  val=55  test=0


## Cell 7 — Datasets & DataLoaders


In [18]:
train_ds = SmartCacheDataset(
    data=train_files, transform=train_transforms,
    cache_rate=CACHE_RATE,
    num_init_workers=CACHE_NUM_WORKERS,
    num_replace_workers=CACHE_NUM_WORKERS,
)
val_ds = SmartCacheDataset(
    data=val_files, transform=val_transforms,
    cache_rate=CACHE_RATE,
    num_init_workers=CACHE_NUM_WORKERS,
    num_replace_workers=CACHE_NUM_WORKERS,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False,
)
val_loader = DataLoader(
    val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False, collate_fn=pad_list_data_collate,
)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


Loading dataset: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 27/27 [01:10<00:00,  2.60s/it]

Train batches : 55
Val batches   : 27


## Cell 8 — Model, Loss, Optimiser, Metrics
- **SwinUNETR**: Shifted-window transformer (feature_size=48, img_size=96³)
- **CombinedNETLoss**: FocalTversky(α=0.3, β=0.7, γ=0.75)×0.60 + WeightedBCE(pos_weight=30)×0.40
- **out_channels=2** (BG=0, NET=1)


In [19]:
# ── Model ─────────────────────────────────────────────────────────────────────
model = SwinUNETRWrapper(
    in_channels       = IN_CHANNELS,
    out_channels      = OUT_CHANNELS,
    img_size          = PATCH_SIZE,
    feature_size      = SWIN_FEATURE_SIZE,
    depths            = SWIN_DEPTHS,
    num_heads         = SWIN_NUM_HEADS,
    drop_rate         = SWIN_DROP_RATE,
    attn_drop_rate    = SWIN_ATTN_DROP,
    dropout_path_rate = SWIN_DROP_PATH,
    use_checkpoint    = True,
).to(DEVICE)

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel across {NUM_GPUS} GPUs")

try:
    model = torch.compile(model, backend="aot_eager")
    print("torch.compile applied (aot_eager)")
except Exception as e:
    print(f"torch.compile skipped: {e}")

n_params = sum(p.numel() for p in model.parameters())
print(f"SwinUNETR parameters: {n_params/1e6:.2f} M")

# ── Loss ──────────────────────────────────────────────────────────────────────
loss_function = CombinedNETLoss(
    ftl_alpha   = FTL_ALPHA,
    ftl_beta    = FTL_BETA,
    ftl_gamma   = FTL_GAMMA,
    pos_weight  = NET_POS_WEIGHT,
    ftl_weight  = COMBINED_FTL_WEIGHT,
    bce_weight  = COMBINED_BCE_WEIGHT,
).to(DEVICE)

# ── Post-processing ───────────────────────────────────────────────────────────
post_label = AsDiscrete(to_onehot=OUT_CHANNELS)
post_pred  = AsDiscrete(argmax=True, to_onehot=OUT_CHANNELS)

# ── Metrics ───────────────────────────────────────────────────────────────────
dice_metric = DiceMetric(
    include_background=False,
    reduction=MetricReduction.MEAN_BATCH,
    get_not_nans=True,
)
hd95_metric = HausdorffDistanceMetric(
    include_background=False, percentile=95.0,
    reduction=MetricReduction.MEAN_BATCH,
)

def _get_raw(m):
    """Unwrap torch.compile / DataParallel to get raw state dict."""
    if hasattr(m, "_orig_mod"): m = m._orig_mod
    if hasattr(m, "module"):    m = m.module
    return m

# ── Optimiser ─────────────────────────────────────────────────────────────────
try:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY, fused=True,
    )
    print("Fused AdamW applied")
except TypeError:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    print("Standard AdamW")

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS,
)

# ── Sliding-window inferer ────────────────────────────────────────────────────
model_inferer = partial(
    sliding_window_inference,
    roi_size=SW_ROI, sw_batch_size=SW_BATCH,
    predictor=model, overlap=SW_OVERLAP,
)

# ── GradScaler ────────────────────────────────────────────────────────────────
scaler = GradScaler(enabled=USE_AMP and torch.cuda.is_available())

print("\nAll components ready ✓")
print(f"  Loss   : CombinedNETLoss  FTL(α={FTL_ALPHA},β={FTL_BETA},γ={FTL_GAMMA})×{COMBINED_FTL_WEIGHT} + BCE(pw={NET_POS_WEIGHT})×{COMBINED_BCE_WEIGHT}")
print(f"  Output : {OUT_CHANNELS} channels → BG(0) | NET(1)")


DataParallel across 2 GPUs
torch.compile applied (aot_eager)
SwinUNETR parameters: 62.19 M
Fused AdamW applied

All components ready ✓
  Loss   : CombinedNETLoss  FTL(α=0.3,β=0.7,γ=0.75)×0.6 + BCE(pw=30.0)×0.4
  Output : 2 channels → BG(0) | NET(1)


## Cell 9 — Training Loop
Saves checkpoint on best **NET Dice**. Early stops after no improvement for `EARLY_STOP_PATIENCE` validation intervals.


In [20]:
best_metric              = -1.0
best_metric_epoch        = -1
epochs_since_improvement = 0
epoch_loss_values        = []
val_metric_values        = []
log_rows                 = []
run_start                = time.time()

# ── Progress widgets ──────────────────────────────────────────────────────────
epoch_bar    = widgets.IntProgress(
    value=0, min=0, max=MAX_EPOCHS,
    description="Epochs:", bar_style="info",
    layout=widgets.Layout(width="80%"),
)
status_label = widgets.Label(value="Starting...")
val_label    = widgets.Label(value="")
display(widgets.VBox([epoch_bar, status_label, val_label]))

# ── Training loop ─────────────────────────────────────────────────────────────
cc_dice = float("nan")   # initialise for log rows before first val step

for epoch in range(1, MAX_EPOCHS + 1):

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss  = 0.0
    n_steps     = 0
    epoch_start = time.time()

    for batch_data in train_loader:
        inputs = torch.cat(
            [batch_data[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS],
            dim=1,
        )
        labels = batch_data["label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP):
            outputs = model(inputs)
            loss    = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        n_steps    += 1

    epoch_loss /= max(n_steps, 1)
    epoch_loss_values.append(epoch_loss)
    scheduler.step()

    elapsed = time.time() - epoch_start
    lr_now  = optimizer.param_groups[0]["lr"]

    epoch_bar.value    = epoch
    status_label.value = (
        f"Epoch {epoch}/{MAX_EPOCHS}  |  Loss={epoch_loss:.4f}  "
        f"|  lr={lr_now:.2e}  |  {elapsed:.1f}s"
    )
    print(f"Epoch {epoch:03d}/{MAX_EPOCHS} | Loss: {epoch_loss:.4f} | lr: {lr_now:.2e} | {elapsed:.1f}s")

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    if epoch % VAL_INTERVAL == 0:
        model.eval()
        val_start = time.time()

        with torch.no_grad():
            for vb in val_loader:
                vi = torch.cat(
                    [vb[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS],
                    dim=1,
                )
                vl = vb["label"].to(DEVICE, non_blocking=True)

                with autocast(enabled=USE_AMP):
                    vo_logits = model_inferer(vi)

                vo_list = decollate_batch(vo_logits)
                vl_list = decollate_batch(vl)
                vo_bin  = [post_pred(p)  for p in vo_list]
                vl_bin  = [post_label(l) for l in vl_list]
                dice_metric(y_pred=vo_bin, y=vl_bin)

        dice_agg, _ = dice_metric.aggregate()
        dice_metric.reset()
        net_dice = float(dice_agg.mean().item())
        val_metric_values.append({"epoch": epoch, "NET_dice": net_dice})

        improved_marker = ""
        if net_dice > best_metric:
            best_metric              = net_dice
            best_metric_epoch        = epoch
            epochs_since_improvement = 0
            raw = _get_raw(model)
            torch.save(raw.state_dict(), RUN_DIR / BEST_MODEL_NAME)
            torch.save(
                raw.state_dict(),
                RUN_DIR / "checkpoints" / f"net_epoch{epoch:03d}_dice{net_dice:.4f}.pth"
            )
            improved_marker     = "  *** NEW BEST ***"
            epoch_bar.bar_style = "success"
        else:
            epochs_since_improvement += 1
            epoch_bar.bar_style = "info"

        val_elapsed = time.time() - val_start
        val_label.value = (
            f"Val @ ep {epoch}  |  NET Dice={net_dice:.4f}  "
            f"|  best={best_metric:.4f}@ep{best_metric_epoch}  "
            f"|  {val_elapsed:.1f}s{improved_marker}"
        )
        print(f" ---> [VAL] NET Dice: {net_dice:.4f} | Best: {best_metric:.4f} @ Ep {best_metric_epoch}{improved_marker}")
        cc_dice = net_dice  # reuse variable name for logging

        # Early stopping
        if (EARLY_STOP_PATIENCE is not None
                and epochs_since_improvement >= EARLY_STOP_PATIENCE):
            print(f"\nEarly stop @ epoch {epoch} — best NET Dice={best_metric:.4f} @ ep {best_metric_epoch}")
            epoch_bar.bar_style = "warning"
            break

    # ── Save log every epoch ──────────────────────────────────────────────────
    log_rows.append({
        "epoch":         epoch,
        "lr":            lr_now,
        "train_loss":    epoch_loss,
        "val_NET_dice":  cc_dice if epoch % VAL_INTERVAL == 0 else float("nan"),
        "epoch_seconds": round(elapsed, 1),
    })
    pd.DataFrame(log_rows).to_csv(RUN_DIR / "training_log.csv", index=False)

total_min = (time.time() - run_start) / 60
status_label.value = (
    f"Done — {total_min:.1f} min  |  best NET Dice={best_metric:.4f} @ ep {best_metric_epoch}"
)
epoch_bar.bar_style = "success"
print(f"\nTraining complete — {total_min:.1f} min | Best NET Dice: {best_metric:.4f} @ epoch {best_metric_epoch}")


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 8.19 MiB is free. Process 16531 has 15.22 GiB memory in use. Process 29264 has 15.51 GiB memory in use. Including non-PyTorch memory, this process has 1014.00 MiB memory in use. Of the allocated memory 620.22 MiB is allocated by PyTorch, and 29.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Cell 10 — Training Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(range(1, len(epoch_loss_values)+1), epoch_loss_values,
             lw=1.5, color="steelblue")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Combined Loss")
axes[0].set_title("Training Loss — NET Subregion 3D UNet")
axes[0].grid(alpha=0.3)

if val_metric_values:
    val_df = pd.DataFrame(val_metric_values)
    axes[1].plot(val_df["epoch"], val_df["NET_dice"], "o-",
                 lw=1.5, color="steelblue", label="NET Dice")
    if best_metric_epoch > 0:
        axes[1].axvline(best_metric_epoch, ls="--", c="black", alpha=0.5,
                        label=f"best={best_metric:.3f}@ep{best_metric_epoch}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Val NET Dice (Binary: BG vs NET)")
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out_path = RUN_DIR / "figures" / "training_curves_net.png"
plt.savefig(out_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


## Cell 11 — Test-Set Evaluation


In [ ]:
dice_metric_t = DiceMetric(
    include_background=False, reduction=MetricReduction.MEAN_BATCH, get_not_nans=True,
)
hd95_metric_t = HausdorffDistanceMetric(
    include_background=False, percentile=95.0, reduction=MetricReduction.MEAN_BATCH,
)

test_ds     = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=False, collate_fn=pad_list_data_collate,
)

# Load best checkpoint into a fresh (unwrapped) model
raw_model = SwinUNETRWrapper(
    in_channels       = IN_CHANNELS,
    out_channels      = OUT_CHANNELS,
    img_size          = PATCH_SIZE,
    feature_size      = SWIN_FEATURE_SIZE,
    depths            = SWIN_DEPTHS,
    num_heads         = SWIN_NUM_HEADS,
    use_checkpoint    = False,   # no gradient checkpointing at inference
).to(DEVICE)
raw_model.load_state_dict(torch.load(RUN_DIR / BEST_MODEL_NAME, map_location=DEVICE))
raw_model.eval()

test_inferer = partial(
    sliding_window_inference,
    roi_size=SW_ROI, sw_batch_size=SW_BATCH,
    predictor=raw_model, overlap=SW_OVERLAP,
)

per_subject = []
print("Running NET test-set evaluation...")
print(f"{'Idx':>4}  {'Subject':<25}  {'NET Dice':>9}  {'HD95':>8}")
print("-" * 55)

with torch.no_grad():
    for idx, tb in enumerate(test_loader, 1):
        sid = tb["subject_id"][0]
        ti  = torch.cat([tb[k].to(DEVICE) for k in IMAGE_KEYS], dim=1)
        tl  = tb["label"].to(DEVICE)

        with autocast(enabled=USE_AMP):
            to_logits = test_inferer(ti)

        to_list = decollate_batch(to_logits)
        tl_list = decollate_batch(tl)
        to_bin  = [post_pred(p)  for p in to_list]
        tl_bin  = [post_label(l) for l in tl_list]

        dice_metric_t.reset()
        dice_metric_t(y_pred=to_bin, y=tl_bin)
        net_d = float(dice_metric_t.aggregate()[0].mean().item())

        try:
            hd95_metric_t.reset()
            hd95_metric_t(y_pred=to_bin, y=tl_bin)
            hd95 = float(hd95_metric_t.aggregate().mean().item())
        except Exception:
            hd95 = float("nan")

        pred_label = to_bin[0].argmax(dim=0).cpu().numpy().astype(np.uint8)
        ref_path   = [f for f in test_files if f["subject_id"] == sid][0]["label"]
        ref_img    = nib.load(ref_path)
        nib.save(
            nib.Nifti1Image(pred_label, ref_img.affine),
            str(RUN_DIR / "predictions" / f"{sid}_NET_pred.nii.gz")
        )

        per_subject.append({"subject_id": sid, "dice_NET": net_d, "hd95_mm": hd95})
        print(f"{idx:4d}  {sid:<25}  {net_d:9.4f}  {hd95:8.2f}")

per_df = pd.DataFrame(per_subject).sort_values("dice_NET", ascending=False)
per_df.to_csv(RUN_DIR / "test_metrics_net.csv", index=False)
print(f"\nMean NET Dice  : {per_df['dice_NET'].mean():.4f}")
print(f"Median NET Dice: {per_df['dice_NET'].median():.4f}")
print(f"Saved: {RUN_DIR / 'test_metrics_net.csv'}")


## Cell 12 — Reload Instructions


In [ ]:
print("=" * 60)
print(f" Best NET Dice (val): {best_metric:.4f} @ epoch {best_metric_epoch}")
print(f" Best model saved  : {RUN_DIR / BEST_MODEL_NAME}")
print("=" * 60)
print("""
To reload for inference:

  model = SwinUNETRWrapper(
      in_channels=4, out_channels=2,
      img_size=(96, 96, 96),
      feature_size=48,
      depths=(2, 2, 2, 2),
      num_heads=(3, 6, 12, 24),
      use_checkpoint=False,
  )
  model.load_state_dict(torch.load("best_swinunetr_net.pth"))
  model.eval()

Input : 4-channel MRI (T1n, T1c, T2w, T2f) normalised channel-wise
Output: Binary mask — 0=Background, 1=NET (Non-Enhancing Tumor core)

Loss  : FocalTversky(α=0.3,β=0.7,γ=0.75)×0.60 + WeightedBCE(pw=30)×0.40
          FTL β=0.7  → heavily penalises missed NET voxels (FN)
          BCE pw=30  → prevents class-collapse on low-prevalence NET

Arch  : SwinUNETR shifted-window transformer (feature_size=48)
          Long-range attention across T1n/T1CE contrast difference
          for discriminating NET from adjacent ET/CC subregions
""")
